In [ ]:
!git clone https://github.com/TheAtticusProject/cuad.git


Cloning into 'cuad'...
remote: Enumerating objects: 30, done.
remote: Total 30 (delta 0), reused 0 (delta 0), pack-reused 30 (from 1)
Receiving objects: 100% (30/30), 17.78 MiB | 10.83 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
!ls cuad


category_descriptions.csv  data.zip	readme.md  scrape.py  utils.py
contract_review.png	   evaluate.py	run.sh	   train.py


In [ ]:
!ls cuad


category_descriptions.csv  data.zip	readme.md  scrape.py  utils.py
contract_review.png	   evaluate.py	run.sh	   train.py


In [ ]:
!unzip cuad/data.zip -d cuad/data



Archive:  cuad/data.zip
  inflating: cuad/data/CUADv1.json   
  inflating: cuad/data/test.json     
  inflating: cuad/data/train_separate_questions.json  


In [ ]:
import json

with open("/content/cuad/data/CUADv1.json") as f:
    cuad = json.load(f)

print("Total contracts:", len(cuad["data"]))


Total contracts: 510


In [ ]:
SELECTED_CLAUSES = [
    "Termination",
    "Governing Law",
    "Confidentiality",
    "Limitation of Liability",
    "Indemnification",
    "Payment",
    "Intellectual Property",
    "Warranty Disclaimer",
    "Force Majeure",
    "Assignment"
]


In [ ]:
import json

# Load CUAD dataset
with open("/content/cuad/data/CUADv1.json") as f:
    cuad = json.load(f)

print("Total contracts:", len(cuad["data"]))

# Collect all unique clause (question) names
clause_names = set()

for doc in cuad["data"]:
    for para in doc["paragraphs"]:
        for qa in para["qas"]:
            clause_names.add(qa["question"].strip())

print("Total unique clause types:", len(clause_names))

# Show first 20 clause names
list(clause_names)[:20]


Total contracts: 510
Total unique clause types: 41


['Highlight the parts (if any) of this contract related to "Exclusivity" that should be reviewed by a lawyer. Details: Is there an exclusive dealing\xa0 commitment with the counterparty? This includes a commitment to procure all “requirements” from one party of certain technology, goods, or services or a prohibition on licensing or selling technology, goods or services to third parties, or a prohibition on\xa0 collaborating or working with other parties), whether during the contract or\xa0 after the contract ends (or both).',
 'Highlight the parts (if any) of this contract related to "License Grant" that should be reviewed by a lawyer. Details: Does the contract contain a license granted by one party to its counterparty?',
 'Highlight the parts (if any) of this contract related to "Most Favored Nation" that should be reviewed by a lawyer. Details: Is there a clause that if a third party gets better terms on the licensing or sale of technology/goods/services described in the contract, t

In [ ]:
def extract_clause_name(question):
    """
    Extracts the clause name inside quotes from CUAD question text
    """
    if '"' in question:
        return question.split('"')[1]
    return question.strip()


In [ ]:
# Test on a few clause questions
sample_questions = list(clause_names)[:5]

for q in sample_questions:
    print("RAW :", q)
    print("CLEAN:", extract_clause_name(q))
    print("-" * 60)


RAW : Highlight the parts (if any) of this contract related to "Exclusivity" that should be reviewed by a lawyer. Details: Is there an exclusive dealing  commitment with the counterparty? This includes a commitment to procure all “requirements” from one party of certain technology, goods, or services or a prohibition on licensing or selling technology, goods or services to third parties, or a prohibition on  collaborating or working with other parties), whether during the contract or  after the contract ends (or both).
CLEAN: Exclusivity
------------------------------------------------------------
RAW : Highlight the parts (if any) of this contract related to "License Grant" that should be reviewed by a lawyer. Details: Does the contract contain a license granted by one party to its counterparty?
CLEAN: License Grant
------------------------------------------------------------
RAW : Highlight the parts (if any) of this contract related to "Most Favored Nation" that should be reviewed b

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=40,
    problem_type="multi_label_classification"
)

model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
text = "This Agreement shall be governed by the laws of the State of California."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)


In [ ]:
import torch

with torch.no_grad():
    outputs = model(**inputs)

probs = torch.sigmoid(outputs.logits)
print(probs)


tensor([[0.4271, 0.5185, 0.5520, 0.4679, 0.3830, 0.4661, 0.6148, 0.6932, 0.4952,
         0.4591, 0.5598, 0.4899, 0.5900, 0.6615, 0.4571, 0.5138, 0.5063, 0.6215,
         0.5773, 0.5845, 0.5611, 0.4683, 0.4973, 0.4160, 0.5140, 0.5692, 0.5689,
         0.5287, 0.4573, 0.5879, 0.3978, 0.3626, 0.4214, 0.5002, 0.5709, 0.4631,
         0.5128, 0.5434, 0.3358, 0.4477]])


In [ ]:
labels = [
 'Affiliate License-Licensee',
 'Affiliate License-Licensor',
 'Agreement Date',
 'Anti-Assignment',
 'Audit Rights',
 'Cap On Liability',
 'Change Of Control',
 'Competitive Restriction Exception',
 'Covenant Not To Sue',
 'Document Name',
 'Effective Date',
 'Exclusivity',
 'Expiration Date',
 'Governing Law',
 'Insurance',
 'Ip Ownership Assignment',
 'Irrevocable Or Perpetual License',
 'Joint Ip Ownership',
 'License Grant',
 'Liquidated Damages',
 'Minimum Commitment',
 'Most Favored Nation',
 'No-Solicit Of Customers',
 'No-Solicit Of Employees',
 'Non-Compete',
 'Non-Disparagement',
 'Non-Transferable License',
 'Notice Period To Terminate Renewal',
 'Parties',
 'Post-Termination Services',
 'Price Restrictions',
 'Renewal Term',
 'Revenue/Profit Sharing',
 'Rofr/Rofo/Rofn',
 'Source Code Escrow',
 'Termination For Convenience',
 'Third Party Beneficiary',
 'Uncapped Liability',
 'Unlimited/All-You-Can-Eat-License',
 'Volume Restriction',
 'Warranty Duration'
]

# convert tensor → list
scores = probs[0].tolist()

label_scores = list(zip(labels, scores))

for label, score in label_scores:
    print(f"{label:35s} -> {score:.3f}")


Affiliate License-Licensee          -> 0.427
Affiliate License-Licensor          -> 0.519
Agreement Date                      -> 0.552
Anti-Assignment                     -> 0.468
Audit Rights                        -> 0.383
Cap On Liability                    -> 0.466
Change Of Control                   -> 0.615
Competitive Restriction Exception   -> 0.693
Covenant Not To Sue                 -> 0.495
Document Name                       -> 0.459
Effective Date                      -> 0.560
Exclusivity                         -> 0.490
Expiration Date                     -> 0.590
Governing Law                       -> 0.662
Insurance                           -> 0.457
Ip Ownership Assignment             -> 0.514
Irrevocable Or Perpetual License    -> 0.506
Joint Ip Ownership                  -> 0.622
License Grant                       -> 0.577
Liquidated Damages                  -> 0.584
Minimum Commitment                  -> 0.561
Most Favored Nation                 -> 0.468
No-Solicit

In [ ]:
THRESHOLD = 0.5

predicted_clauses = [
    label for label, score in label_scores if score >= THRESHOLD
]

print("Predicted clauses:")
for c in predicted_clauses:
    print("•", c)


Predicted clauses:
• Affiliate License-Licensor
• Agreement Date
• Change Of Control
• Competitive Restriction Exception
• Effective Date
• Expiration Date
• Governing Law
• Ip Ownership Assignment
• Irrevocable Or Perpetual License
• Joint Ip Ownership
• License Grant
• Liquidated Damages
• Minimum Commitment
• Non-Compete
• Non-Disparagement
• Non-Transferable License
• Notice Period To Terminate Renewal
• Post-Termination Services
• Rofr/Rofo/Rofn
• Source Code Escrow
• Third Party Beneficiary
• Uncapped Liability


In [ ]:
TARGET_CLAUSES = [
    "Governing Law",
    "Termination For Convenience",
    "Confidentiality",            # maps to Non-Disparagement + NDA-like
    "Limitation Of Liability",    # maps to Cap On Liability / Uncapped Liability
    "Indemnification",
    "License Grant",
    "Non-Compete",
    "Non-Transferable License",
    "Payment Terms",              # maps to Revenue/Profit Sharing / Minimum Commitment
    "Intellectual Property Ownership"  # maps to IP Ownership clauses
]


In [ ]:
import json

with open("cuad/data/CUADv1.json") as f:
    cuad = json.load(f)

print("Total contracts:", len(cuad["data"]))


Total contracts: 510


In [ ]:
# Map your project clauses to CUAD labels
CLAUSE_MAP = {
    "Governing Law": ["Governing Law"],
    "Termination": ["Termination For Convenience"],
    "Confidentiality": ["Non-Disparagement"],
    "Limitation Of Liability": ["Cap On Liability", "Uncapped Liability"],
    "License Grant": ["License Grant"],
    "Non-Compete": ["Non-Compete"],
    "Assignment": ["Anti-Assignment"],
    "IP Ownership": ["Ip Ownership Assignment", "Joint Ip Ownership"],
    "Payment Terms": ["Revenue/Profit Sharing", "Minimum Commitment"],
}


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
samples = []

for contract in cuad["data"]:
    context = contract["paragraphs"][0]["context"]
    sentences = sent_tokenize(context)

    # Collect all annotations
    annotations = []
    for qa in contract["paragraphs"][0]["qas"]:
        question = qa["question"]
        answers = qa.get("answers", [])
        annotations.append((question, answers))

    for sent in sentences:
        sent_lower = sent.lower()
        label_vector = {k: 0 for k in CLAUSE_MAP.keys()}

        for proj_clause, cuad_labels in CLAUSE_MAP.items():
            for q, answers in annotations:
                for cuad_label in cuad_labels:
                    if cuad_label.lower() in q.lower():
                        for ans in answers:
                            if ans["text"].lower() in sent_lower:
                                label_vector[proj_clause] = 1

        # Keep sentence only if it has at least one positive label
        if sum(label_vector.values()) > 0:
            samples.append({
                "sentence": sent,
                **label_vector
            })

print("Total extracted samples:", len(samples))
print(samples[0])


Total extracted samples: 4205
{'sentence': "Company  hereby   appoints                            Distributor as Company's exclusive distributor within                            the Market and grants to  Distributor  the  exclusive                            right  to sell and  distribute  Products  within  the                            Market,   and   Distributor   hereby   accepts   such                            appointment  and such grant,  in accordance  with the                            terms and conditions of this  Agreement.", 'Governing Law': 0, 'Termination': 0, 'Confidentiality': 0, 'Limitation Of Liability': 0, 'License Grant': 1, 'Non-Compete': 0, 'Assignment': 0, 'IP Ownership': 0, 'Payment Terms': 0}


In [ ]:
import random

negative_samples = []
positive_sentences = set([s["sentence"] for s in samples])

for contract in cuad["data"]:
    context = contract["paragraphs"][0]["context"]
    sentences = sent_tokenize(context)

    for sent in sentences:
        if sent not in positive_sentences:
            negative_samples.append({
                "sentence": sent,
                "Governing Law": 0,
                "Termination": 0,
                "Confidentiality": 0,
                "Limitation Of Liability": 0,
                "License Grant": 0,
                "Non-Compete": 0,
                "Assignment": 0,
                "IP Ownership": 0,
                "Payment Terms": 0,
            })

# balance dataset
negative_samples = random.sample(
    negative_samples,
    min(len(negative_samples), len(samples))
)

print("Negative samples:", len(negative_samples))


Negative samples: 4205


In [ ]:
final_dataset = samples + negative_samples
random.shuffle(final_dataset)

print("Final dataset size:", len(final_dataset))
print(final_dataset[0])


Final dataset size: 8410
{'sentence': "All Prefilled Syringes in Antares' possession shall be subject to disposition by AMAG upon expiration or termination of this Agreement, and in either such event, Antares or its Subcontractor shall deliver the Prefilled Syringes to AMAG or its designee, at AMAG's\n\n- 10 -\n\n\n\n\n\n[***] INDICATES MATERIAL THAT HAS BEEN OMITTED AND FOR WHICH CONFIDENTIAL TREATMENT HAS BEEN REQUESTED.", 'Governing Law': 0, 'Termination': 0, 'Confidentiality': 0, 'Limitation Of Liability': 0, 'License Grant': 0, 'Non-Compete': 0, 'Assignment': 0, 'IP Ownership': 0, 'Payment Terms': 0}


In [ ]:
import re

def clean_sentence(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[\*\*\*].*?\[\*\*\*]', '', text)
    return text.strip()


In [ ]:
for row in final_dataset:
    row["sentence"] = clean_sentence(row["sentence"])

print(final_dataset[0])


{'sentence': "All Prefilled Syringes in Antares' possession shall be subject to disposition by AMAG upon expiration or termination of this Agreement, and in either such event, Antares or its Subcontractor shall deliver the Prefilled Syringes to AMAG or its designee, at AMAG's - 10 - [***] INDICATES MATERIAL THAT HAS BEEN OMITTED AND FOR WHICH CONFIDENTIAL TREATMENT HAS BEEN REQUESTED.", 'Governing Law': 0, 'Termination': 0, 'Confidentiality': 0, 'Limitation Of Liability': 0, 'License Grant': 0, 'Non-Compete': 0, 'Assignment': 0, 'IP Ownership': 0, 'Payment Terms': 0}


In [ ]:
import pandas as pd

df = pd.DataFrame(final_dataset)
df.to_csv("cuad_clause_dataset.csv", index=False)

print("Saved dataset with shape:", df.shape)


Saved dataset with shape: (8410, 10)


In [ ]:
import pandas as pd

df = pd.read_csv("cuad_clause_dataset.csv")

TEXT_COLUMN = "sentence"
LABEL_COLUMNS = [
    "Governing Law",
    "Termination",
    "Confidentiality",
    "Limitation Of Liability",
    "License Grant",
    "Non-Compete",
    "Assignment",
    "IP Ownership",
    "Payment Terms"
]

texts = df[TEXT_COLUMN].tolist()
labels = df[LABEL_COLUMNS].values

print("Total samples:", len(texts))
print("Label shape:", labels.shape)
print("Sample text:", texts[0])
print("Sample label:", labels[0])


Total samples: 8410
Label shape: (8410, 9)
Sample text: All Prefilled Syringes in Antares' possession shall be subject to disposition by AMAG upon expiration or termination of this Agreement, and in either such event, Antares or its Subcontractor shall deliver the Prefilled Syringes to AMAG or its designee, at AMAG's - 10 - [***] INDICATES MATERIAL THAT HAS BEEN OMITTED AND FOR WHICH CONFIDENTIAL TREATMENT HAS BEEN REQUESTED.
Sample label: [0 0 0 0 0 0 0 0 0]


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class LegalClauseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }


In [ ]:
dataset = LegalClauseDataset(texts, labels, tokenizer)

sample = dataset[0]
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"])


torch.Size([256])
torch.Size([256])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [ ]:
from torch.utils.data import DataLoader, random_split

# create dataset
full_dataset = LegalClauseDataset(texts, labels, tokenizer)

# split sizes
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size]
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

# dataloaders
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


Train size: 6728
Validation size: 1682


In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

NUM_LABELS = 9

model = AutoModelForSequenceClassification.from_pretrained(
    "nlpaueb/legal-bert-base-uncased",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded on:", device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on: cuda


In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ---- Training ----
    model.train()
    total_train_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_train_loss = total_train_loss / len(train_loader)
    print("Average training loss:", avg_train_loss)

    # ---- Validation ----
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print("Average validation loss:", avg_val_loss)



Epoch 1/3


100%|██████████| 841/841 [04:53<00:00,  2.87it/s]


Average training loss: 0.13813022109286752
Average validation loss: 0.06408926104394039

Epoch 2/3


100%|██████████| 841/841 [04:54<00:00,  2.86it/s]


Average training loss: 0.04985147865996314
Average validation loss: 0.04920077107634872

Epoch 3/3


100%|██████████| 841/841 [04:54<00:00,  2.85it/s]


Average training loss: 0.03281298233943634
Average validation loss: 0.04407628364552042


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from google.colab import files

uploaded = files.upload()   # Upload any legal PDF
pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)


Saving mou-legal.pdf to mou-legal.pdf
Uploaded file: mou-legal.pdf


In [ ]:
!pip install pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 109.2 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

raw_text = extract_text_from_pdf(pdf_path)
print(raw_text[:1000])  # preview


This Memorandum of Understanding is executed between the parties with the intention of 
defining the framework under which they shall collaborate for the proposed commercial 
engagement. The parties acknowledge that this document records their mutual intent, 
expectations, and responsibilities and is entered into in good faith for the purpose of fostering 
cooperation and operational alignment.The parties agree that the relationship established under 
this understanding is based on mutual trust, transparency, and professional competence. Each 
party represents that it possesses the necessary expertise, resources, and authority to carry out 
the activities contemplated herein and that entering into this understanding does not violate any 
existing contractual or legal obligations.One party shall be responsible for sourcing, 
manufacturing, and supplying the products or services required for execution of the project, 
while the other party shall undertake responsibilities relating to ten

In [ ]:
import re

def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 30]
    return sentences

sentences = split_into_sentences(raw_text)
print("Total sentences:", len(sentences))
print(sentences[:3])


Total sentences: 26
['This Memorandum of Understanding is executed between the parties with the intention of \ndefining the framework under which they shall collaborate for the proposed commercial \nengagement.', 'The parties acknowledge that this document records their mutual intent, \nexpectations, and responsibilities and is entered into in good faith for the purpose of fostering \ncooperation and operational alignment.The parties agree that the relationship established under \nthis understanding is based on mutual trust, transparency, and professional competence.', 'Each \nparty represents that it possesses the necessary expertise, resources, and authority to carry out \nthe activities contemplated herein and that entering into this understanding does not violate any \nexisting contractual or legal obligations.One party shall be responsible for sourcing, \nmanufacturing, and supplying the products or services required for execution of the project, \nwhile the other party shall unde

In [ ]:
LABELS = [
    "Governing Law",
    "Termination",
    "Confidentiality",
    "Limitation Of Liability",
    "License Grant",
    "Non-Compete",
    "Assignment",
    "IP Ownership",
    "Payment Terms"
]


In [ ]:
import torch
import numpy as np

def predict_sentence(sentence, threshold=0.5):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    results = []
    for label, score in zip(LABELS, probs):
        if score >= threshold:
            results.append((label, float(score)))

    return results


In [ ]:
results = []

for sent in sentences:
    preds = predict_sentence(sent)
    if preds:
        results.append({
            "sentence": sent,
            "predictions": preds
        })

len(results)


26

In [ ]:
for r in results[:10]:
    print("Sentence:", r["sentence"])
    for label, score in r["predictions"]:
        print(f"  → {label} ({score:.2f})")
    print("-" * 80)


Sentence: This Memorandum of Understanding is executed between the parties with the intention of 
defining the framework under which they shall collaborate for the proposed commercial 
engagement.
  → Governing Law (0.58)
  → Termination (0.61)
  → Limitation Of Liability (0.58)
  → Assignment (0.51)
--------------------------------------------------------------------------------
Sentence: The parties acknowledge that this document records their mutual intent, 
expectations, and responsibilities and is entered into in good faith for the purpose of fostering 
cooperation and operational alignment.The parties agree that the relationship established under 
this understanding is based on mutual trust, transparency, and professional competence.
  → Governing Law (0.51)
  → Termination (0.59)
  → Limitation Of Liability (0.61)
  → Assignment (0.55)
--------------------------------------------------------------------------------
Sentence: Each 
party represents that it possesses the necessary

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

os.listdir("/content/drive/MyDrive")


['Colab Notebooks',
 'inbound4153501838830243364.mp4',
 'DOC-20250120-WA0010..pdf',
 'Untitled document (4).gdoc',
 'Untitled presentation.gslides',
 'internrep (4).niv.gdoc',
 'internrep (3).niv.gdoc',
 'internrep (2).niv.gdoc',
 'internrep (1).niv.gdoc',
 'internrep.niv.gdoc',
 '23CS068-HindujaaS (1) (1).gdoc',
 'Document 64.gdoc',
 'Untitled document (3).gdoc',
 'SignCloud: Serverless Real-Time Sign Language Translator for Virtual Classrooms.docx',
 'SignCloud: Serverless Real-Time Sign Language Translator for Virtual Classrooms (1).gdoc',
 'SignCloud: Serverless Real-Time Sign Language Translator for Virtual Classrooms.gdoc',
 'sih (1).ppt1.gslides',
 'sih.ppt1.gslides',
 'PERU.gslides',
 'sih.ppt.gslides',
 'SIHfinalppt.gslides',
 'Untitled document (2).gdoc',
 'K M ASHIQUR-RAHMAN resume.gdoc',
 'Document 64 (14).gdoc',
 'Screenshot_20251125_133217_GPay.jpg',
 'Untitled document (1).gdoc',
 'Untitled document.gdoc',
 'PDF_to_Audiobook_Converter_Report.pdf',
 'Divya Internship repo

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=9,
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("✅ tokenizer and model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✅ tokenizer and model loaded
